In [ ]:
!pip install -U pandas numpy scikit-learn matplotlib seaborn

In [ ]:
# =====================================================
# IMPORTAÇÕES
# =====================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
# =====================================================
# CARREGAR DADOS
# =====================================================

train_url = "https://raw.githubusercontent.com/elsweb/mba2026/main/07-aprendizado-de-maquina/kaggle/playground-series-s4e2/train.csv"
test_url = "https://raw.githubusercontent.com/elsweb/mba2026/main/07-aprendizado-de-maquina/kaggle/playground-series-s4e2/test.csv"

train = pd.read_csv(train_url)
test = pd.read_csv(test_url)

print("Train:", train.shape)
print("Test:", test.shape)

train.head()

In [ ]:
# =====================================================
# >>> FEATURE ENGINEERING (IMC)
# =====================================================

train["BMI"] = train["Weight"] / (train["Height"] ** 2)
test["BMI"] = test["Weight"] / (test["Height"] ** 2)

In [ ]:
# =====================================================
# VISÃO GERAL
# =====================================================

target = "NObeyesdad"

plt.figure()
sns.countplot(data=train, y=target, order=train[target].value_counts().index)
plt.title("Distribuição das Classes (Target)")
plt.show()

In [ ]:
# =====================================================
# VARIÁVEL ALVO
# =====================================================

X = train.drop(columns=[target, "id"])
y = train[target]

In [ ]:
# =====================================================
# IDENTIFICAR COLUNAS
# =====================================================

categorical_features = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["number"]).columns.tolist()

print("Categóricas:", categorical_features)
print("Numéricas:", numerical_features)

In [ ]:
# =====================================================
# DISTRIBUIÇÃO DAS VARIÁVEIS NUMÉRICAS
# =====================================================

X[numerical_features].hist(bins=20, figsize=(15, 10))
plt.suptitle("Distribuição das Variáveis Numéricas")
plt.show()

In [ ]:
# =====================================================
# >>> BOXPLOTS
# =====================================================

for col in numerical_features:
    plt.figure(figsize=(8,2))
    sns.boxplot(x=X[col])
    plt.title(f"Boxplot - {col}")
    plt.show()

In [ ]:
# =====================================================
# CORRELAÇÃO ENTRE VARIÁVEIS
# =====================================================

#O mapa de calor apresenta a correlação entre as variáveis numéricas.
#Observa-se que a maioria das correlações é fraca, enquanto Height e Weight apresentam uma correlação positiva mais significativa.

plt.figure(figsize=(12,8))
sns.heatmap(
    X[numerical_features].corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlação entre Variáveis Numéricas")
plt.show()

In [ ]:
# =====================================================
# >>> GRÁFICOS DE DISPERSÃO
# =====================================================

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=train,
    x="Height",
    y="Weight",
    hue="NObeyesdad",
    alpha=0.7
)

plt.title("Altura x Peso")
plt.show()

In [ ]:
# =====================================================
# >>> HISTOGRAMA DO IMC
# =====================================================

plt.figure(figsize=(8,5))
sns.histplot(train["BMI"], bins=30, kde=True)

plt.title("Distribuição do IMC")
plt.show()

In [ ]:
# =====================================================
# >>> IMC POR CLASSE
# =====================================================

plt.figure(figsize=(12,6))
sns.boxplot(
    data=train,
    x="NObeyesdad",
    y="BMI"
)

plt.xticks(rotation=45)
plt.title("Distribuição do IMC por Classe")
plt.show()

In [ ]:
# =====================================================
# PRÉ-PROCESSAMENTO
# =====================================================

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

In [ ]:
# =====================================================
# DIVISÃO TREINO/VALIDAÇÃO
# =====================================================

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# =====================================================
# RANDOM FOREST
# =====================================================

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", model)
])

In [ ]:
# =====================================================
# TREINAMENTO
# =====================================================

pipeline.fit(X_train, y_train)

In [ ]:
# =====================================================
# AVALIAÇÃO
# =====================================================

preds = pipeline.predict(X_valid)

print("Accuracy:", accuracy_score(y_valid, preds))
print(classification_report(y_valid, preds))

In [ ]:
# =====================================================
# CONFUSION MATRIX (VISUAL IMPORTANTE)
# =====================================================

cm = confusion_matrix(y_valid, preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()

In [ ]:
# =====================================================
# TREINAR COM TODOS OS DADOS
# =====================================================

pipeline.fit(X, y)

In [ ]:
# =====================================================
# PREVISÕES TESTE
# =====================================================

test_predictions = pipeline.predict(test)

In [ ]:
# =====================================================
# FEATURE IMPORTANCE (MUITO IMPORTANTE PRA APRESENTAÇÃO)
# =====================================================

ohe = pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

cat_features = ohe.get_feature_names_out(categorical_features)
all_features = np.concatenate([numerical_features, cat_features])

importances = pipeline.named_steps["classifier"].feature_importances_

feat_df = pd.DataFrame({
    "feature": all_features,
    "importance": importances
}).sort_values(by="importance", ascending=False).head(15)

plt.figure(figsize=(10,6))
sns.barplot(data=feat_df, x="importance", y="feature")
plt.title("Top 15 Features Mais Importantes")
plt.show()

In [ ]:
# =====================================================
# FEATURE IMPORTANCE (MUITO IMPORTANTE PRA APRESENTAÇÃO)
# =====================================================

test_predictions = pipeline.predict(test)

In [ ]:
# =====================================================
# SUBMISSION KAGGLE
# =====================================================

submission_url = "https://raw.githubusercontent.com/elsweb/mba2026/main/07-aprendizado-de-maquina/kaggle/playground-series-s4e2/sample_submission.csv"


submission = pd.DataFrame({
    "id": test["id"],
    "NObeyesdad": test_predictions
})

submission.to_csv(
    submission_url,
    index=False
)

print("\nArquivo submission.csv gerado com sucesso!")